# Conversion for Mobile Deployment


- PyTorch → ONNX (in Prototype 10)
- ONNX → TensorFlow → TFLite + INT8 Quantization

References
- https://github.com/DanieliusKr/onnx-example/blob/main/onnx_example.ipynb
- https://www.geeksforgeeks.org/machine-learning/convert-pytorch-model-to-tf-lite-with-onnx-tf/

### Import

In [1]:
# Check Python version
!python --version

Python 3.11.13


In [2]:
try:
    import google.colab
    IN_COLAB = True

    !pip install onnx2tf
    """
    !pip install onnx
    !pip install onnx_tf

    !pip install tensorflow-addons
    !pip install tensorflow-probability
    """
    from google.colab import files
except ImportError:
    IN_COLAB = False

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.5/17.5 MB 54.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.0/5.0 MB 62.2 MB/s eta 0:00:00
  Attempting uninstall: ml_dtypes
    Found existing installation: ml-dtypes 0.4.1
    Uninstalling ml-dtypes-0.4.1:
      Successfully uninstalled ml-dtypes-0.4.1
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
tensorflow 2.18.0 requires ml-dtypes<0.5.0,>=0.4.0, but you have ml-dtypes 0.5.4 which is incompatible.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 226.1/226.1 kB 18.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 611.8/611.8 kB 41.1 MB/s eta 0:00:00
  Attempting uninstall: typeguard
    Found existing installation: typeguard 4.4.4
    Uninstalling typeguard-4.4.4:
      Successfully uninstalled typeguard-4.4.4
ERROR: pip's dependency resolver does not currently take into account all t

In [5]:
import torch as torch
import torch.nn as nn
import os
import random
import numpy as np
import cv2
import pandas as pd

import onnx2tf
import tensorflow as tf
import torch.onnx
import onnx
# from onnx_tf.backend import prepare

ModuleNotFoundError: No module named 'keras.engine'

## Conversion to TensorFlow and TFLite

In [ ]:
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
HEIGHT = 128
WIDTH = 128

def data_gen():
    # Initialize dataset folder
    !unzip -q 'Dataset_AIGD.zip'
    dataset = ""
    get_input(DEVICE, True, dataset)

def get_input(DEVICE, isRandom, data) -> torch.Tensor:
    """ Generate random input data with the same shape as the model's expected input """
    if isRandom:
        # Generate random video
        # Timesteps, Channels, Height, Width
        video_tensor = torch.rand(20, 3, 1, 1).to(DEVICE)
        # Generate random intent
        intent_direction = random.randint(0, 2)
        # Generate random intent position
        intent_position = random.randint(0, 9)

        # Convert video to frames
        frames = []
        for i in range(video_tensor.shape[0]):
            frame_tensor = video_tensor[i]
            frame_tensor= get_intent_torch(intent_position, i, intent_direction, frame_tensor, 1)
            frames.append(frame_tensor)

        video_tensor = torch.stack(frames, dim=0)
        # Adds batch dimension to the tensor -> Batch, Timesteps, Channels, Height, Width
        video_tensor = video_tensor.unsqueeze(0)

        return video_tensor
    else:
        data_train = os.path.join(data, "train")
        data_train_vid = os.path.join(data_train, "videos")
        data_train_lbl = os.path.join(data_train, "labels")
        labels = [f for f in os.listdir(data_train_lbl) if f.endswith('.csv')]

        for i, video_name in enumerate(os.listdir(data_train_vid)):
        # Representative dataset for Integer Quantization
            if i < 100:
                intent_position = get_intent_position()
                intent = get_intent(intent_position, labels[i])

                video_path = os.path.join(data_train_vid, video_name)
                cap = cv2.VideoCapture(video_path)
                frames = []

                for i in range(20):
                    ret, frame = cap.read()
                    if not ret:
                        frame_tensor = torch.zeros((3, HEIGHT, WIDTH))
                    else:
                        frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
                        """
                        if self.transforms:
                            frame = Image.fromarray(frame)
                            frame_tensor = self.transforms(frame)
                        else:
                        """
                        frame_tensor = torch.from_numpy(frame).permute(2, 0, 1).float() / 255.0

                    frame_tensor = get_intent_torch(intent_position, i, intent, frame_tensor, 128)
                    frames.append(frame_tensor)

            cap.release()
            video_tensor = torch.stack(frames, dim=0)

            # Global Average Pooling to fit the 1x1 dimensions
            global_avg_pool = nn.AdaptiveAvgPool2d((1, 1))
            video_tensor = global_avg_pool(video_tensor)
            representative_data.append(video_tensor)

        return representative_data

def get_intent_position():
    # 50% of the dataset have intent
    if random.random() < 0.5:
        # The time positions of the first 1 second (10 fps) because the intent is placed for a second
        start_frame = 0
        end_frame = 9
        median = (start_frame + end_frame)/2
        range_zero = np.arange(-median, median)

        # Obtain the probability of selecting a timestamp using the adjacent 0.5 areas
        smaller_range = range_zero - 0.5
        higher_range = range_zero + 0.5

        # Probability is the difference of the probability of higher range and lower range
        probability = ss.norm.cdf(higher_range) - ss.norm.cdf(smaller_range)

        # Normalize the probabilities
        # Each probability in probability range is divided by the sum of the probabilities in probability range
        probability /= probability.sum()

        # Select a timestamp based on the probabilities
        range = np.arange(start_frame, end_frame)
        intent_position = np.random.choice(range, p=probability)
    else:
        intent_position = -1

    return intent_position

def get_intent(intent_position, df):
        # Check if the data has no intent
        if intent_position != -1:
            intent = get_label(df)
        else:
            intent = -1
        return intent

def get_label(df):
    """Extract label from CSV using majority vote on last 24 frames."""
    col = 'label_id_corrected' if 'label_id_corrected' in df.columns else df.columns[-1]
    counts = df[col].tail(24).value_counts() # CSV files are in 24 fps

    label_counts = [counts.get(i, 0) for i in range(3)]

    if label_counts[0] == 24:
        return 0  # Front
    elif label_counts[1] > label_counts[2]:
        return 1  # Left
    elif label_counts[1] < label_counts[2]:
        return 2  # Right
    else:
        return int(df[col].tail(12).mode().iloc[0])

def get_intent_torch(intent_position, i, intent, frame_tensor, size):
    # Create a tensor for the intent with the same spatial dimensions as the video frames
    # Used for no intent
    intent_torch = torch.zeros((3, size, size))
    # If intent exists, add intent in its intent position for 1 second (10 frames)
    if intent_position != -1 and intent_position <= i and (intent_position + 10) > i:
        # Convert intent to one-hot vector by filling the specified channel with 1
        intent_torch[intent, :, :] = 1

    intent_torch = intent_torch.to(frame_tensor.device)
    # Append the intent as a channel to the video frame
    frame_tensor = torch.cat((frame_tensor, intent_torch), dim=0)

    return frame_tensor

In [ ]:
"""
# Order of input shape is changed
onnx_path = "convlstm.onnx"
tf_dir = "tf_model"

# Convert from ONNX to TensorFlow and TFLite
onnx2tf.convert(
    input_onnx_file_path=onnx_path,
    output_folder_path=tf_dir,
    copy_onnx_input_output_names_to_tflite=True,
    keep_ncw_or_nchw_or_ncdhw_input_names=["video_input"], # (Did not work)
    # output_dynamic_range_quantized_tflite=True,
    output_integer_quantized_tflite = True,
    non_verbose=True,
)
print(f"Success: conversion to Tensorflow. File is saved at: {tf_dir}")
"""

Success: conversion to Tensorflow. File is saved at: tf_model
  adding: content/tf_model/ (stored 0%)
  adding: content/tf_model/convlstm_float32.tflite (deflated 14%)
  adding: content/tf_model/convlstm_float16.tflite (deflated 20%)
  adding: content/tf_model/variables/ (stored 0%)
  adding: content/tf_model/variables/variables.index (deflated 33%)
  adding: content/tf_model/variables/variables.data-00000-of-00001 (deflated 82%)
  adding: content/tf_model/fingerprint.pb (stored 0%)
  adding: content/tf_model/convlstm_dynamic_range_quant.tflite (deflated 21%)
  adding: content/tf_model/saved_model.pb (deflated 11%)
  adding: content/tf_model/schema.fbs (deflated 71%)
  adding: content/tf_model/schema_generated.py (deflated 92%)
  adding: content/tf_model/assets/ (stored 0%)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

TensorFlow and TFLite models downloaded ✅


In [ ]:
# Outdated method for conversion to TensorFlow
onnx_path = "convlstm.onnx"
tf_dir = "tf_model"

onnx_model = onnx.load(onnx_path)
tf_model = prepare(onnx_model)

os.makedirs(tf_dir, exist_ok=True)
tf_model.export_graph(tf_dir)
print(f"Success: conversion to Tensorflow. File is saved at: {tf_dir}")

In [ ]:
# Convert the TensorFlow model to TFLite model
converter = tf.lite.TFLiteConverter.from_saved_model(tf_dir)
# Dynamic Range Quantization: INT 8 Quantization
converter.optimizations = [tf.lite.Optimize.DEFAULT]
tflite_model = converter.convert()

# Save the TFLite model
with open ("convlstm.tflite", "wb") as f:
    f.write(tflite_model)

tflite_path = os.path.join(tf_dir, "convlstm.tflite")
print(f"Success: conversion to TFLite. File is saved at: {tflite_path}")

In [ ]:
if IN_COLAB:
    # Automated Download TensorFlow and TFLite models
    if os.path.exists(tf_dir):
        !zip -r 'tf_model.zip' '/content/tf_model'
        files.download("tf_model.zip")
        print("TensorFlow and TFLite models downloaded ✅")
    else:
        print("TensorFlow and TFLite models were not downloaded.")

In [ ]:
# Verifying the TFLite model
tflite_path = os.path.join(tf_dir, "convlstm_float16.tflite")
interpreter = tf.lite.Interpreter(model_path=tflite_path)
interpreter.allocate_tensors()

output_details = interpreter.get_output_details()[0] # Model has single output.
input_details = interpreter.get_input_details()[0] # Model has single input.
input_shape = input_details['shape']

# Input shape: Batch size, Channels, Height, Width, Timesteps
# Input_data: Batch size, Timesteps, Channels, Height, Width
input_data = get_input(DEVICE, True, None).cpu().numpy().astype(np.float32)
input_tflite = np.transpose(input_data, (0, 2, 3, 4, 1))
interpreter.set_tensor(input_details['index'], input_tflite)
interpreter.invoke()

output = interpreter.get_tensor(output_details['index'])
output_shape = output.shape

print("Interpreted TFLite model")
print(f"Input shape: {input_shape}")
print(f"Output shape: {output_shape}")
print(f"Output: {output}")

Interpreted TFLite model
Input shape: [ 1  6  1  1 20]
Output shape: (1, 3)
Output: [[0.07523145 0.01889999 0.06085416]]
